# Data Preparation from Scratch
## From SIMA annual workbooks to analysis data, step by step

This notebook documents the complete data-building process for reproducibility: reading the ORIGINAL SIMA workbooks, consolidating physical stations into monitoring zones, counting missing observations, filling gaps through two alternative methods, and constructing the OFFICIAL school calendar. The calendar is fixed by date and taken from SEP agreements published in the Diario Oficial de la Federación; it is not inferred from the dataset. This notebook also provides a recipe for incorporating future SIMA workbooks.

**Data access.** The hourly measurements belong to SIMA (Sistema Integral de Monitoreo Ambiental, Gobierno del Estado de Nuevo León). SIMA provided the annual workbooks for this study on the condition that they are used in the analysis but not redistributed. This repository therefore contains no measurement series at any resolution: neither the workbooks in `data_raw/` nor the consolidated hourly, station, zone, day-window or gap files built by notebooks 00 and 01. These are all git-ignored. The repository contains the complete code, official SEP school calendar, trained MIMA model and all derived results reported in the article: estimates, diagnostics, tables and figures.

To rebuild everything, request the hourly data from SIMA (https://aire.nl.gob.mx/) and place the workbooks in `data_raw/` as `BD 2023.xlsx`, `BD 2024.xlsx` and `BD 2025.xlsx`. Each workbook must have one sheet per station, named with its SIMA code; the first column must be `Fecha y hora` or `date`, followed by one column per parameter. Run this notebook; it stops with an explanatory message if inputs are missing. Then produce the MIMA series with `data_processed/mima/mima_pipeline.py`. Original Spanish input column names and station codes are retained because they are part of the provider's data format.

**Role in the analysis.** Data preparation: the pipeline from the SIMA workbooks to the consolidated hourly bases, the imputations and the official calendar. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

raw_dir = Path('../data_raw')
output_dir = Path('../data_processed')
censo_dir = output_dir / 'censo_datos'
censo_dir.mkdir(parents=True, exist_ok=True)

print('Libraries imported successfully')

Libraries imported successfully


## 2. Reading the ORIGINAL SIMA workbooks

The three annual workbooks (`BD 2023.xlsx`, `BD 2024.xlsx`, `BD 2025.xlsx`) share a format: one sheet per physical station, with the SIMA sector code as sheet name, a timestamp in the first column and one column per parameter (CO, NO, NO2, NOX, O3, PM10, PM2.5, PRS, RAINF, RH, SO2, SR, TOUT, WSR, WDR). The analysis uses the five pollutants covered throughout the network: CO, NO2, O3, PM10 and PM2.5.

All records are converted to long format: source file, physical station, timestamp, parameter and value. The original machine-readable field names are retained for compatibility.

In [2]:
# Workbook sheet keys -> physical-station names (original SIMA codes)
codigos_estacion = {'SE': 'SURESTE', 'CE': 'CENTRO', 'SO': 'SUROESTE', 'NE': 'NORESTE',
                    'NO': 'NOROESTE', 'NTE': 'NORTE', 'NE2': 'NORESTE2', 'SE2': 'SURESTE2',
                    'SE3': 'SURESTE 3', 'NO2': 'NOROESTE 2', 'NO3': 'NOROESTE 3',
                    'NTE2': 'NORTE 2', 'SO2': 'SUROESTE2', 'SUR': 'SUR', 'NE3': 'NORESTE 3'}

# Physical station -> monitoring zone (SIMA sector families)
mapeo_zona = {'SUR': 'SUR', 'SURESTE': 'SUR', 'SURESTE2': 'SUR', 'SURESTE 3': 'SUR',
              'SUROESTE': 'SUROESTE2', 'SUROESTE2': 'SUROESTE2',
              'NORTE': 'NORTE2', 'NORTE 2': 'NORTE2',
              'CENTRO': 'CENTRO',
              'NORESTE': 'NORESTE3', 'NORESTE2': 'NORESTE3', 'NORESTE 3': 'NORESTE3',
              'NOROESTE': 'NOROESTE3', 'NOROESTE 2': 'NOROESTE3', 'NOROESTE 3': 'NOROESTE3'}


def leer_libro_sima(ruta, nombre_archivo):
    "\n    Read an annual SIMA workbook (one sheet per physical station) in long format.\n    Normalize headers to allow units and alternative timestamp headers\n    ('Fecha y hora' or 'date') in the original workbooks.\n    "
    hojas = pd.read_excel(ruta, sheet_name=None, engine='calamine')
    piezas = []
    for hoja, df in hojas.items():
        df = df.copy()
        df.columns = [str(c).split(' (')[0].strip() for c in df.columns]
        df = df.rename(columns={'Fecha y hora': 'date', 'fecha': 'date', 'Fecha': 'date'})
        if 'date' not in df.columns:
            continue
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['date'])
        presentes = [c for c in contaminantes if c in df.columns]
        largo = df.melt(id_vars=['date'], value_vars=presentes,
                        var_name='parametro', value_name='valor')
        largo['valor'] = pd.to_numeric(largo['valor'], errors='coerce')
        clave = hoja.strip().upper()
        largo['estacion_fisica'] = codigos_estacion.get(clave, clave)
        piezas.append(largo)
    todo = pd.concat(piezas, ignore_index=True)
    todo['archivo'] = nombre_archivo
    return todo


archivos_crudos = {'2023': raw_dir / 'BD 2023.xlsx',
                   '2024': raw_dir / 'BD 2024.xlsx',
                   '2025': raw_dir / 'BD 2025.xlsx'}
hay_crudos = all(ruta.exists() for ruta in archivos_crudos.values())

print("=" * 70)
print('READING ORIGINAL WORKBOOKS')
print("=" * 70)
if hay_crudos:
    piezas_crudas = []
    for nombre, ruta in archivos_crudos.items():
        df = leer_libro_sima(ruta, nombre)
        piezas_crudas.append(df)
        print(f"  {nombre}: {len(df):,} rows | {df['estacion_fisica'].nunique()} physical stations | "
              f"{df['date'].min().date()} to {df['date'].max().date()}")
    crudos = pd.concat(piezas_crudas, ignore_index=True)
    crudos['zona'] = crudos['estacion_fisica'].map(mapeo_zona)
    sin_zona = crudos[crudos['zona'].isna()]['estacion_fisica'].unique()
    print(f"  TOTAL: {len(crudos):,} rows (including missing values)")
    print(f"  Physical stations without an assigned zone: {list(sin_zona) if len(sin_zona) else 'none'}")
    # One physical-station row per station, timestamp and parameter. Duplicate
    # timestamps within a workbook are averaged.
    base_estaciones = (crudos.dropna(subset=['valor', 'zona'])
                       .groupby(['estacion_fisica', 'zona', 'date', 'parametro'], as_index=False)['valor'].mean())
    base_estaciones.to_csv(output_dir / 'base_horaria_estaciones_fisicas.csv.gz',
                           index=False, compression='gzip')
    print(f"\nSaved to: {output_dir / 'base_horaria_estaciones_fisicas.csv.gz'}")
else:
    raise FileNotFoundError(
        "The SIMA annual workbooks are not in data_raw/. They are not redistributed: request them from SIMA (https://aire.nl.gob.mx/) and place them as 'BD 2023.xlsx', 'BD 2024.xlsx' and 'BD 2025.xlsx'.")

print(f"\nPhysical-station base: {len(base_estaciones):,} rows | "
      f"{base_estaciones['estacion_fisica'].nunique()} stations | "
      f"{base_estaciones['date'].min()} to {base_estaciones['date'].max()}")

READING ORIGINAL WORKBOOKS


  2023: 656,850 rows | 15 physical stations | 2023-01-01 to 2023-12-31


  2024: 658,705 rows | 15 physical stations | 2024-01-01 to 2024-12-31


  2025: 656,890 rows | 15 physical stations | 2025-01-01 to 2025-12-31


  TOTAL: 1,972,445 rows (including missing values)
  Physical stations without an assigned zone: none



Saved to: ../data_processed/base_horaria_estaciones_fisicas.csv.gz

Physical-station base: 1,751,750 rows | 15 stations | 2023-01-01 00:00:00 to 2025-12-31 23:00:00


## 3. Organizing the network by SECTOR

SIMA names its 15 physical stations by sector families (SURESTE, SURESTE2, SURESTE 3, etc.). Stations within a family monitor the same urban sector; according to the network's characterization, each station represents a radius of roughly 2–3 km. The analysis groups these families into six monitoring zones. Below, we check whether stations within a zone move together, supporting their consolidation, and whether grouping stabilizes coverage when an individual station fails.

In [3]:
tabla_zonas = (base_estaciones.groupby(['zona', 'estacion_fisica'])
               .agg(filas=('valor', 'count'),
                    desde=('date', lambda s: s.min().date()),
                    hasta=('date', lambda s: s.max().date())).reset_index())
print("=" * 70)
print('ZONE COMPOSITION (constituent physical stations)')
print("=" * 70)
print(tabla_zonas.to_string(index=False))
tabla_zonas.to_csv(censo_dir / 'zonas_y_estaciones_fisicas.csv', index=False)
print(f"\nSaved to: {censo_dir / 'zonas_y_estaciones_fisicas.csv'}")

ZONE COMPOSITION (constituent physical stations)
     zona estacion_fisica  filas      desde      hasta
   CENTRO          CENTRO 123007 2023-01-01 2025-12-31
 NORESTE3         NORESTE 123828 2023-01-01 2025-12-31
 NORESTE3       NORESTE 3  99063 2023-01-01 2025-12-31
 NORESTE3        NORESTE2 107096 2023-01-01 2025-12-31
NOROESTE3        NOROESTE 100719 2023-01-01 2025-12-31
NOROESTE3      NOROESTE 2 121082 2023-01-01 2025-12-31
NOROESTE3      NOROESTE 3  88116 2023-01-01 2025-12-31
   NORTE2           NORTE 124771 2023-01-01 2025-12-31
   NORTE2         NORTE 2 126179 2023-01-01 2025-12-31
      SUR             SUR 122107 2023-01-01 2025-12-31
      SUR         SURESTE 120965 2023-01-01 2025-12-31
      SUR       SURESTE 3 123982 2023-01-01 2025-12-31
      SUR        SURESTE2 122467 2023-01-01 2025-12-31
SUROESTE2        SUROESTE 123431 2023-01-01 2025-12-31
SUROESTE2       SUROESTE2 124937 2023-01-01 2025-12-31

Saved to: ../data_processed/censo_datos/zonas_y_estaciones_fisicas.csv

## 4. Consolidating the hourly zone series

For each zone, timestamp and pollutant, we average the physical stations in that zone. We examine within-zone coherence before interpreting these averages. Notebook 02 also reports descriptive sensitivity to using the median instead of the mean: diagnostic differences range from 5.5% to 12.9% across pollutants. These are differences between series, not between regression coefficients; they do not establish that the aggregation choice is inconsequential.

In [4]:
base_horaria = (base_estaciones.groupby(['zona', 'date', 'parametro'], as_index=False)['valor'].mean()
                .rename(columns={'zona': 'estacion'}))

print("=" * 70)
print('HOURLY BASE CONSOLIDATED BY ZONE')
print("=" * 70)
print(f"Rows: {len(base_horaria):,} | zones: {base_horaria['estacion'].nunique()} | "
      f"{base_horaria['date'].min()} to {base_horaria['date'].max()}")

# Within-zone coherence: daily correlations between physical stations.
# Stations grouped within a zone should show coherent temporal variation;
# correlations support consolidation but do not establish identical air masses.
diario_fisico = (base_estaciones
                 .assign(fecha=lambda d: d['date'].dt.normalize())
                 .groupby(['zona', 'estacion_fisica', 'fecha', 'parametro'])['valor'].mean()
                 .reset_index())
print('\nWithin-zone coherence (mean daily correlation between stations in each zone):')
coherencia = []
for zona_v, sub_z in diario_fisico.groupby('zona'):
    if sub_z['estacion_fisica'].nunique() < 2:
        print(f"  {zona_v}: 1 station (no pair to compare)")
        continue
    correlaciones = []
    for c in contaminantes:
        ancho = (sub_z[sub_z['parametro'] == c]
                 .pivot_table(index='fecha', columns='estacion_fisica', values='valor'))
        if ancho.shape[1] >= 2:
            m = ancho.corr().values
            correlaciones.append(m[np.triu_indices_from(m, k=1)].mean())
    print(f"  {zona_v}: mean pairwise correlation = {np.nanmean(correlaciones):.2f}")
    coherencia.append({'zona': zona_v, 'n_estaciones': sub_z['estacion_fisica'].nunique(),
                       'correlacion_media_pares': round(float(np.nanmean(correlaciones)), 3)})
pd.DataFrame(coherencia).to_csv(censo_dir / 'coherencia_intrazona.csv', index=False)

base_horaria.to_csv(output_dir / 'base_horaria_zonas.csv.gz', index=False, compression='gzip')
print(f"\nSaved to: {output_dir / 'base_horaria_zonas.csv.gz'}")

HOURLY BASE CONSOLIDATED BY ZONE
Rows: 772,426 | zones: 6 | 2023-01-01 00:00:00 to 2025-12-31 23:00:00



Within-zone coherence (mean daily correlation between stations in each zone):
  CENTRO: 1 station (no pair to compare)
  NORESTE3: mean pairwise correlation = 0.57
  NOROESTE3: mean pairwise correlation = 0.58
  NORTE2: mean pairwise correlation = 0.68


  SUR: mean pairwise correlation = 0.66
  SUROESTE2: mean pairwise correlation = 0.66



Saved to: ../data_processed/base_horaria_zonas.csv.gz


## 5. Missing-data census: how much is missing and how much is imputed?

We count missing observations twice: by zone, which is the level imputed for analysis, and by physical monitor before consolidation. The difference explains one benefit of grouping stations: a zone-hour is missing only when every monitor in that zone is missing.

In [5]:
fecha_min, fecha_max = base_horaria['date'].min(), base_horaria['date'].max()
horas_esperadas = len(pd.date_range(fecha_min, fecha_max, freq='h'))

presentes = (base_horaria.groupby(['estacion', 'parametro'])['valor'].count()
             .unstack('parametro').reindex(columns=contaminantes))
tabla_faltantes = pd.DataFrame({
    'horas_esperadas': horas_esperadas,
    **{f'faltante_{c}_pct': ((1 - presentes[c] / horas_esperadas) * 100).round(1) for c in contaminantes}})

print("=" * 70)
print(f"MISSING DATA BY ZONE AND POLLUTANT ({fecha_min.date()} to {fecha_max.date()})")
print("=" * 70)
print(tabla_faltantes.to_string())
promedio_faltante = np.mean([(1 - presentes[c] / horas_esperadas).mean() for c in contaminantes]) * 100
print(f"\nOverall mean missing hours (filled by imputation): {promedio_faltante:.1f}%")
tabla_faltantes.to_csv(censo_dir / 'tabla_datos_perdidos.csv')
print(f"Saved to: {censo_dir / 'tabla_datos_perdidos.csv'}")

# Repeat the missingness count at the physical-monitor level before consolidation.
# A zone-hour is missing only when ALL its monitors are missing.
presentes_fisicas = (base_estaciones.groupby(['estacion_fisica', 'parametro'])['valor'].count()
                     .unstack('parametro').reindex(columns=contaminantes))
faltante_fisicas = ((1 - presentes_fisicas / horas_esperadas) * 100).round(1)
faltante_fisicas.index.name = 'estacion_fisica'
print('\nMISSING DATA BY PHYSICAL MONITOR AND POLLUTANT (before zone consolidation)')
print(faltante_fisicas.to_string())
promedio_faltante_fisicas = float(faltante_fisicas.values.mean())
print(f"\nMean at physical-monitor level: {promedio_faltante_fisicas:.1f}% | "
      f"at zone level (filled by imputation): {promedio_faltante:.1f}%")
faltante_fisicas.to_csv(censo_dir / 'pct_faltante_por_estacion_fisica.csv')
print(f"Saved to: {censo_dir / 'pct_faltante_por_estacion_fisica.csv'}")

MISSING DATA BY ZONE AND POLLUTANT (2023-01-01 to 2025-12-31)
           horas_esperadas  faltante_CO_pct  faltante_NO2_pct  faltante_O3_pct  faltante_PM10_pct  faltante_PM2.5_pct
estacion                                                                                                             
CENTRO               26304              8.1               3.1              3.9                2.3                15.0
NORESTE3             26304              1.2               3.0              0.2                0.1                 2.9
NOROESTE3            26304              0.2               1.0              0.7                0.1                14.2
NORTE2               26304              0.2               0.2              0.7                0.3                 1.1
SUR                  26304              0.1               0.1              0.1                0.1                 0.1
SUROESTE2            26304              0.3               0.1              0.6                0.2               


MISSING DATA BY PHYSICAL MONITOR AND POLLUTANT (before zone consolidation)
parametro          CO   NO2    O3  PM10  PM2.5
estacion_fisica                               
CENTRO            8.1   3.1   3.9   2.3   15.0
NORESTE           5.6   5.2   6.3   3.5    8.7
NORESTE 3        10.7  10.0   5.7   5.4   91.6
NORESTE2         16.5  15.3   6.9   4.4   49.7
NOROESTE         14.4  12.4   8.2   4.0   78.1
NOROESTE 2        2.6  11.7   3.7   2.4   19.2
NOROESTE 3       19.1  29.5  16.7  13.1   86.5
NORTE             3.0   3.0   5.2   4.4   10.1
NORTE 2           2.5   1.1   4.5   2.4    9.8
SUR               7.5   4.7   1.6   1.9   20.1
SURESTE          13.6   1.9   2.9   2.7   19.0
SURESTE 3         9.5   2.5   2.2   2.2   12.3
SURESTE2         14.6   3.3   7.9   2.7    5.9
SUROESTE          2.8   4.1   5.0   2.7   16.3
SUROESTE2         4.2   0.7   1.6   4.6   14.0

Mean at physical-monitor level: 11.2% | at zone level (filled by imputation): 2.1%
Saved to: ../data_processed/censo_datos/p

## 6. Filling gaps through two alternative methods

The analysis is run with two imputation methods to assess sensitivity to how missing values are filled.

**(a) Hybrid seasonal-local imputation, implemented here.** For each zone–pollutant series, temporal interpolation fills up to six consecutive interior missing values, following the local trajectory. This can partially fill longer gaps. Remaining gaps are filled using the series' median profile for the relevant month, day type (weekday or weekend) and hour. Profiles are calculated after interpolation. If a stratum has no data, the code falls back to the month-by-hour profile. Pollutant-specific floors are then applied to the whole series, including observed values. These implementation details matter when interpreting the missing-data diagnostics.

**(b) MIMA.** This deep-learning imputation approach was developed for the network by Gómez, Sánchez, Medina, Montibeller and Pérez (Tecnológico de Monterrey). The supplied implementation produces a SAITS-based series; its CSDI stage does not change the effective imputation examined here. The script `../data_processed/mima/mima_pipeline.py` trains on the consolidated hourly base, saves the model and audits recovery of artificially hidden observed cells. Notebook 04 examines that internal diagnostic. Training takes time on a CPU, so this notebook loads the series already produced by the script rather than retraining it.

In [6]:
pisos_fisicos = {'CO': 0.05, 'NO2': 0.05, 'O3': 0.5, 'PM10': 1.0, 'PM2.5': 0.5}
malla_horas = pd.date_range(fecha_min, fecha_max, freq='h')

series_imputadas = []
for (zona_v, c), sub_s in base_horaria.groupby(['estacion', 'parametro']):
    s = sub_s.set_index('date')['valor'].reindex(malla_horas)
    observada = s.notna()
    # Step 1: temporal interpolation, up to six consecutive missing values
    s = s.interpolate(method='time', limit=6, limit_area='inside')
    # Step 2: remaining gaps from month x day type x hour profiles
    marco = pd.DataFrame({'valor': s})
    marco['mes'] = marco.index.month
    marco['habil'] = (marco.index.dayofweek < 5).astype(int)
    marco['hora'] = marco.index.hour
    perfil = (marco.dropna(subset=['valor'])
              .groupby(['mes', 'habil', 'hora'])['valor'].median())
    faltan = marco['valor'].isna()
    relleno = marco.loc[faltan, ['mes', 'habil', 'hora']].apply(
        lambda fila: perfil.get((fila['mes'], fila['habil'], fila['hora']), np.nan), axis=1)
    marco.loc[faltan, 'valor'] = relleno
    # Fallback: within-series month x hour profile
    if marco['valor'].isna().any():
        perfil2 = marco.dropna(subset=['valor']).groupby(['mes', 'hora'])['valor'].median()
        faltan2 = marco['valor'].isna()
        marco.loc[faltan2, 'valor'] = marco.loc[faltan2, ['mes', 'hora']].apply(
            lambda fila: perfil2.get((fila['mes'], fila['hora']), np.nan), axis=1)
    marco['valor'] = marco['valor'].clip(lower=pisos_fisicos[c])
    salida_s = marco[['valor']].reset_index(names='date')
    salida_s['estacion'] = zona_v
    salida_s['parametro'] = c
    salida_s['imputada'] = (~observada.values).astype(int)
    series_imputadas.append(salida_s)

imputado_hibrido = pd.concat(series_imputadas, ignore_index=True)

# Final network fallback for empty strata: pollutant month x hour profile
if imputado_hibrido['valor'].isna().any():
    perfil_red = (imputado_hibrido.dropna(subset=['valor'])
                  .assign(mes=lambda d: d['date'].dt.month, hora=lambda d: d['date'].dt.hour)
                  .groupby(['parametro', 'mes', 'hora'])['valor'].median())
    faltan_red = imputado_hibrido['valor'].isna()
    aux = imputado_hibrido.loc[faltan_red].assign(mes=lambda d: d['date'].dt.month,
                                                  hora=lambda d: d['date'].dt.hour)
    imputado_hibrido.loc[faltan_red, 'valor'] = aux.apply(
        lambda fila: perfil_red.get((fila['parametro'], fila['mes'], fila['hora']), np.nan), axis=1)

imputado_hibrido.to_csv(output_dir / 'datos_horarios_imputados_hibrido.csv.gz',
                        index=False, compression='gzip')
ruta_mima = output_dir / 'datos_horarios_imputados_MIMA.csv.gz'
if ruta_mima.exists():
    imputado_mima = pd.read_csv(ruta_mima, parse_dates=['date'])
    series_para_resumen = [('hybrid (this notebook)', imputado_hibrido), ('MIMA', imputado_mima)]
else:
    imputado_mima = None
    series_para_resumen = [('hybrid (this notebook)', imputado_hibrido)]
    print('The MIMA-imputed series does not exist yet. With the zone base saved (section 3), run:')
    print('  cd data_processed/mima && python mima_pipeline.py --reuse-model   (uses the included trained model;')
    print('  without the flag it retrains from scratch in about two CPU hours). Notebook 01 needs it.')
    print('The MIMA-imputed series does not exist yet. With the zone base saved (section 3), run:')
    print("  cd data_processed/mima && python mima_pipeline.py --reuse-model   (uses the included trained model;")
    print("  without the flag it retrains from scratch in about two CPU hours). Notebook 01 needs it.")

print("=" * 70)
print('IMPUTED HOURLY SERIES')
print("=" * 70)
for nombre, df in series_para_resumen:
    cobertura = df['valor'].notna().mean() * 100
    print(f"  {nombre}: {len(df):,} rows | {df['date'].min()} to {df['date'].max()} | "
          f"coverage of the complete grid: {cobertura:.1f}%")
pct_imputado = imputado_hibrido['imputada'].mean() * 100
print(f"  Hours imputed by the hybrid method: {pct_imputado:.1f}% (the remainder is observed)")
print(f"\nSaved to: {output_dir / 'datos_horarios_imputados_hibrido.csv.gz'}")

IMPUTED HOURLY SERIES
  hybrid (this notebook): 789,120 rows | 2023-01-01 00:00:00 to 2025-12-31 23:00:00 | coverage of the complete grid: 100.0%
  MIMA: 789,120 rows | 2023-01-01 00:00:00 to 2025-12-31 23:00:00 | coverage of the complete grid: 100.0%
  Hours imputed by the hybrid method: 2.1% (the remainder is observed)

Saved to: ../data_processed/datos_horarios_imputados_hibrido.csv.gz


## 7. The OFFICIAL school calendar: dated rules and sources

The school-term indicator comes from SEP's official basic-education calendar, NOT from the dataset. Dates are taken from agreements published in the Diario Oficial de la Federación (DOF), including each school year's calendar:

- **2022–23** (Agreement 09/06/22, DOF June 3, 2022): return from winter vacation on January 9, 2023; Easter break April 3–14, 2023; last school day July 26, 2023. Suspensions: February 6, March 17 (administrative work), March 20, May 1, May 5 and May 15, 2023.
- **2023–24** (Agreement 02/06/23, DOF June 26, 2023): starts August 28, 2023; winter break December 18, 2023–January 5, 2024, with return January 8; Easter break March 25–April 5, 2024; last school day July 16, 2024. Suspensions: November 2 and 20, 2023; February 5, March 18, May 1 and May 15, 2024.
- **2024–25** (Agreement 10/06/24, DOF June 10, 2024): starts August 26, 2024; winter break December 19, 2024–January 8, 2025, with return January 9; Easter break April 14–25, 2025; last school day July 16, 2025. Suspensions: September 16 and November 18, 2024; February 3, March 17, May 1, May 5 and May 15, 2025.
- **2025–26** (Agreement 18/06/25, DOF June 9, 2025): starts September 1, 2025; winter break begins December 22, 2025 and continues beyond the study window. Suspensions: September 16 and November 17, 2025.

Summer breaks between school years count as vacation. Monthly teacher-council Fridays, when students do not attend, are not separately flagged. This is a documented limitation: the indicator measures the calendar regime, not actual attendance. The direction of resulting bias is not established.

In [7]:
fechas_malla = pd.date_range(fecha_min.normalize(), fecha_max.normalize(), freq='D')
calendario = pd.DataFrame({'date': fechas_malla})

ciclos_lectivos = [('2023-01-09', '2023-07-26'),   # 2022-23 (partial study window)
                   ('2023-08-28', '2024-07-16'),   # 2023-24
                   ('2024-08-26', '2025-07-16'),   # 2024-25
                   ('2025-09-01', '2026-07-15')]   # 2025-26 (through the end of the study window)
vacaciones = [('2023-04-03', '2023-04-14'),        # Easter 2023
              ('2023-12-18', '2024-01-05'),        # Winter 2023-24
              ('2024-03-25', '2024-04-05'),        # Easter 2024
              ('2024-12-19', '2025-01-08'),        # Winter 2024-25
              ('2025-04-14', '2025-04-25'),        # Easter 2025
              ('2025-12-22', '2026-01-06')]        # Winter 2025-26
suspensiones = ['2023-02-06', '2023-03-17', '2023-03-20', '2023-05-01', '2023-05-05',
                '2023-05-15', '2023-11-02', '2023-11-20', '2024-02-05', '2024-03-18',
                '2024-05-01', '2024-05-15', '2024-09-16', '2024-11-18', '2025-02-03',
                '2025-03-17', '2025-05-01', '2025-05-05', '2025-05-15', '2025-09-16',
                '2025-11-17']

en_ciclo = np.zeros(len(calendario), dtype=bool)
for inicio, fin in ciclos_lectivos:
    en_ciclo |= (calendario['date'] >= inicio) & (calendario['date'] <= fin)
en_vacaciones = np.zeros(len(calendario), dtype=bool)
for inicio, fin in vacaciones:
    en_vacaciones |= (calendario['date'] >= inicio) & (calendario['date'] <= fin)

calendario['lectivo'] = (en_ciclo & ~en_vacaciones).astype(int)
calendario['suspension'] = calendario['date'].isin(pd.to_datetime(suspensiones)).astype(int)
calendario['dow'] = calendario['date'].dt.dayofweek
calendario['dia_clases'] = ((calendario['lectivo'] == 1) & (calendario['dow'] < 5)
                            & (calendario['suspension'] == 0)).astype(int)

def clasificar_tipo_dia(fila):
    '\n    Four categories: weekday/weekend x school term/vacation.\n    '
    habil = fila['dow'] < 5
    lectivo = fila['lectivo'] == 1
    if habil and lectivo:
        return 'habil_lectivo'
    if habil and not lectivo:
        return 'habil_vacaciones'
    if not habil and lectivo:
        return 'finde_lectivo'
    return 'finde_vacaciones'

calendario['tipo_dia'] = calendario.apply(clasificar_tipo_dia, axis=1)

print("=" * 70)
print('OFFICIAL SEP CALENDAR BUILT')
print("=" * 70)
print(f"Study window: {fechas_malla.min().date()} to {fechas_malla.max().date()} ({len(calendario)} days)")
print(f"School-term dates (blocks): {int(calendario['lectivo'].sum())} | "
      f"attendance-proxy days (weekdays, excluding suspensions): {int(calendario['dia_clases'].sum())}")
print('\nCounts by day type:')
print(calendario['tipo_dia'].value_counts())

calendario.to_csv(output_dir / 'calendario_oficial_sep.csv', index=False)
print(f"\nSaved to: {output_dir / 'calendario_oficial_sep.csv'}")

print('\nCOMPLETED')

OFFICIAL SEP CALENDAR BUILT
Study window: 2023-01-01 to 2025-12-31 (1096 days)
School-term dates (blocks): 884 | attendance-proxy days (weekdays, excluding suspensions): 607

Counts by day type:
tipo_dia
habil_lectivo       628
finde_lectivo       256
habil_vacaciones    155
finde_vacaciones     57
Name: count, dtype: int64

Saved to: ../data_processed/calendario_oficial_sep.csv

COMPLETED
